# mp3 → srt → traducción
1. **Transcribe** cada `.mp3` sin `.srt` con faster-whisper, usando tiempos por palabra.
2. **Traduce** cada `.srt` sin su versión `.spa.srt` con Claude, sin tocar los tiempos.

Requisitos: entorno con GPU (*Entorno de ejecución → Cambiar tipo → T4 GPU*) y, para traducir,
el secreto `ANTHROPIC_API_KEY` en el panel 🔑 con acceso habilitado para este notebook.

In [ ]:
!pip install -q faster-whisper anthropic

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/Multimedia/Translations'
TRANSCRIBIR = True
TRADUCIR = True
WHISPER_MODEL = "medium"           # validado; en GPU puedes probar "large-v3" o "large-v3-turbo"
IDIOMA_AUDIO = "en"
IDIOMA_DESTINO = "español latinoamericano neutro"
SUFIJO = "spa"                     # salida: nombre.spa.srt
NOTAS = ""                         # opcional: título, época, personajes, glosario...
USAR_FICHA = True                  # lee todo el srt antes y arma ficha de personajes y trato

## Código (se escribe a archivos .py y se importa)

In [ ]:
%%writefile transcribe.py
"""Transcripción a SRT con tiempos ajustados por palabra.

faster-whisper con timestamps por palabra; los subtítulos se arman a partir de las
palabras (no de los segmentos de Whisper, que vienen redondeados y absorben silencios/música).
"""
import re
import subprocess
import numpy as np
from faster_whisper import WhisperModel

# Parámetros de armado de subtítulos
MAX_CHARS = 84        # máx. caracteres por subtítulo (2 líneas de 42)
LINE_CHARS = 42
MAX_DUR = 6.0         # máx. duración de un subtítulo (s)
MIN_DUR = 1.0         # mín. duración en pantalla (s)
GAP_SPLIT = 0.6       # silencio entre palabras que fuerza corte (s)
PAD_END = 0.8         # cola extra al final de cada subtítulo (s)
MAX_WORD = 1.0        # una palabra no "dura" más que esto (corrige palabras estiradas sobre música)


def cargar_audio(path, sr=16000):
    """Decodifica con ffmpeg a 16 kHz mono float32."""
    out = subprocess.run(["ffmpeg", "-nostdin", "-i", path, "-f", "s16le", "-ac", "1", "-ar", str(sr), "-"],
                         capture_output=True, check=True).stdout
    return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0


def transcribir(model, audio, language="en", **kw):
    if isinstance(audio, str):
        audio = cargar_audio(audio)
    segments, _ = model.transcribe(
        audio, language=language, word_timestamps=True,
        # Sin VAD: Silero se pierde diálogos mezclados con música/ruido en películas.
        # Este filtro descarta alucinaciones rodeadas de silencio (necesita word_timestamps).
        hallucination_silence_threshold=2.0,
        condition_on_previous_text=False,   # evita alucinaciones/bucles en películas largas
        beam_size=5, **kw)
    words = []
    for seg in segments:
        for w in seg.words or []:
            if w.word.strip():
                words.append([w.start, w.end, w.word])
    return words


def _limpiar(words):
    """Recorta palabras anormalmente largas (típico: primera palabra tras música)."""
    for w in words:
        if w[1] - w[0] > MAX_WORD:
            if w[2].strip()[-1:] in ".?!,":
                w[1] = w[0] + MAX_WORD     # palabra final de frase: recortar el final
            else:
                w[0] = w[1] - MAX_WORD     # si no: el inicio está adelantado
    return words


def armar_cues(words):
    words = _limpiar(words)
    cues, cur = [], []

    def flush():
        if cur:
            cues.append([cur[0][0], cur[-1][1], "".join(w[2] for w in cur).strip()])
            cur.clear()

    for i, w in enumerate(words):
        if cur:
            text = "".join(x[2] for x in cur).strip()
            gap = w[0] - cur[-1][1]
            prev_end_punct = cur[-1][2].strip()[-1:] in ".?!"
            too_long = len(text) + len(w[2]) > MAX_CHARS or w[1] - cur[0][0] > MAX_DUR
            if gap > GAP_SPLIT or too_long or (prev_end_punct and (gap > 0.3 or len(text) > 30)):
                flush()
        cur.append(w)
    flush()

    # Ajustes de duración sin pisar el siguiente subtítulo
    for i, c in enumerate(cues):
        nxt = cues[i + 1][0] if i + 1 < len(cues) else c[1] + 10
        c[1] = min(max(c[1] + PAD_END, c[0] + MIN_DUR), nxt - 0.05)
    return cues


def _partir_lineas(text):
    if len(text) <= LINE_CHARS:
        return text
    mid = len(text) // 2
    cands = [m.start() for m in re.finditer(" ", text)]
    cut = min(cands, key=lambda p: abs(p - mid)) if cands else mid
    return text[:cut].strip() + "\n" + text[cut:].strip()


def _ts(t):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def escribir_srt(cues, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, e, t) in enumerate(cues, 1):
            f.write(f"{i}\n{_ts(s)} --> {_ts(e)}\n{_partir_lineas(t)}\n\n")


def main():
    import argparse, time
    ap = argparse.ArgumentParser(description="mp3/mp4 → srt con tiempos por palabra")
    ap.add_argument("audio", nargs="+")
    ap.add_argument("--model", default="medium", help="tiny/base/small/medium/large-v3/large-v3-turbo")
    ap.add_argument("--language", default="en")
    ap.add_argument("--device", default="auto", help="cuda/cpu/auto")
    a = ap.parse_args()
    device = a.device
    if device == "auto":
        import ctranslate2
        device = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
    model = WhisperModel(a.model, device=device, compute_type="float16" if device == "cuda" else "int8")
    for audio in a.audio:
        out = audio.rsplit(".", 1)[0] + ".srt"
        t = time.time()
        escribir_srt(armar_cues(transcribir(model, audio, language=a.language)), out)
        print(f"{out} ({time.time() - t:.0f}s)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile translate.py
"""Traduce un .srt con Claude manteniendo numeración y tiempos intactos.

1. Lee el .srt completo y arma una ficha de la obra (personajes, relaciones, trato de tú o
   usted, glosario, errores de transcripción). Se guarda en nombre.ficha.md para revisarla o
   editarla; si ya existe, se reutiliza.
2. Traduce en bloques (con algunas líneas previas como contexto y la ficha en el prompt) y pide
   la respuesta en JSON con un esquema fijo, así cada traducción vuelve con su id.
Requiere la variable de entorno ANTHROPIC_API_KEY.
"""
import json
import os
import re

import anthropic

MODEL = "claude-opus-5-5"
EFFORT = "medium"     # low/medium/high: más alto = más cuidadoso, más lento y caro
BATCH = 60            # subtítulos por llamada
CONTEXT = 10          # subtítulos previos (ya traducidos) que se envían como contexto
LINE_CHARS = 42

USO = {"input": 0, "output": 0}   # tokens acumulados en esta sesión

BLOCK = re.compile(r"(\d+)\s*\n(\d+:\d+:\d+[,.]\d+\s*-->\s*\d+:\d+:\d+[,.]\d+)[^\n]*\n(.*?)(?=\n\s*\n|\Z)", re.S)

SYSTEM = """Eres un traductor profesional de subtítulos de cine y series.
Traduce cada subtítulo al {idioma}.

Reglas:
- Traduce el sentido y el registro (coloquial, formal, insultos, humor), no palabra por palabra.
- Un subtítulo traducido por cada id recibido, con el mismo id. No unas ni partas subtítulos,
  aunque una frase continúe en el siguiente: tradúcela de forma que se lea bien repartida.
- Sé conciso: el texto debe poder leerse en el tiempo que dura el subtítulo. Máximo 2 líneas
  de unos {chars} caracteres; usa \\n para separar líneas.
- No omitas contenido: todo lo que se dice en el original debe estar en la traducción.
- Si en un mismo subtítulo hablan dos personajes, pon cada uno en su línea con guion
  ("-¿Corriendo?\n-Sí."). Mantén las etiquetas de formato (<i>, </i>, <b>...).
- Los nombres propios no se traducen.
- Si un subtítulo es solo un sonido o una interjección, adáptalo ("Oh" → "Oh", "Huh?" → "¿Eh?").
- El texto viene de un reconocimiento de voz automático y puede tener errores: si una palabra
  no tiene sentido, usa el contexto para inferir lo que se dijo.
- Respeta la ficha de la obra (si se incluye): nombres, trato de tú/usted y glosario.
{notas}{ficha}"""

FICHA_PROMPT = """Abajo está la transcripción automática completa de los subtítulos de una obra
(película o serie), que luego se traducirá al {idioma} en bloques. Prepara una ficha breve que el
traductor tendrá a la vista en cada bloque, para mantener la coherencia de toda la obra.
{notas}
Incluye, en markdown y sin repetir el diálogo:
- **Obra**: de qué trata, época y lugar, tono (y título si se puede deducir).
- **Personajes**: nombre (con la grafía correcta), quién es, edad aproximada y forma de hablar.
- **Relaciones y trato**: quién se trata de tú y quién de usted (o de "señor/señora") en el
  {idioma}, y cómo cambia si cambia a lo largo de la obra.
- **Glosario**: lugares, apodos, términos recurrentes y cómo traducirlos (o si no se traducen).
- **Errores de transcripción**: palabras que el reconocimiento de voz escuchó mal y lo que
  probablemente se dijo (por ejemplo, un nombre escrito de dos formas).

Sé concreto y breve (máximo ~500 palabras). Si algo no se puede deducir, omítelo.

<subtitulos>
{texto}
</subtitulos>"""

SCHEMA = {
    "type": "object",
    "properties": {
        "subtitulos": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"id": {"type": "integer"}, "texto": {"type": "string"}},
                "required": ["id", "texto"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["subtitulos"],
    "additionalProperties": False,
}


def cliente():
    """Cliente con ANTHROPIC_API_KEY; ANTHROPIC_WORKSPACE_ID solo si la key no tiene workspace."""
    ws = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    return anthropic.Anthropic(default_headers={"anthropic-workspace-id": ws} if ws else None)


def leer_srt(path):
    raw = open(path, "rb").read()
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")
    text = text.replace("\r\n", "\n")
    return [{"id": int(n), "tiempo": t.strip(), "texto": tx.strip()} for n, t, tx in BLOCK.findall(text)]


def _rearmar(texto):
    """Respeta los saltos del modelo, salvo que alguna línea quede demasiado larga."""
    lineas = [l.strip() for l in texto.replace("\\n", "\n").split("\n") if l.strip()]
    if all(len(l) <= LINE_CHARS + 6 for l in lineas) and len(lineas) <= 2:
        return "\n".join(lineas)
    if any(l.startswith("-") for l in lineas):          # diálogo: no mezclar hablantes
        return "\n".join(lineas)
    plano = " ".join(lineas)
    if len(plano) <= LINE_CHARS:
        return plano
    mid = len(plano) // 2
    cut = min((m.start() for m in re.finditer(" ", plano)), key=lambda p: abs(p - mid), default=mid)
    return plano[:cut].strip() + "\n" + plano[cut:].strip()


def escribir_srt(subs, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, s in enumerate(subs, 1):
            f.write(f"{i}\n{s['tiempo']}\n{s['texto']}\n\n")


def _llamar(client, **kw):
    """Una llamada a Claude; acumula tokens y valida stop_reason. Devuelve el texto."""
    kw = dict(model=MODEL, max_tokens=16000, **kw)
    if "haiku" in MODEL:                        # Haiku no tiene modelo de respaldo
        response = client.messages.create(**kw)
    else:
        # Si el modelo rechaza la petición, la API la reintenta con el modelo de respaldo recomendado.
        response = client.beta.messages.create(**kw, betas=["server-side-fallback-2026-07-01"],
                                               fallbacks="default")
    USO["input"] += response.usage.input_tokens
    USO["output"] += response.usage.output_tokens
    if response.stop_reason == "refusal":
        cat = response.stop_details.category if response.stop_details else None
        raise RuntimeError(f"Claude rechazó la petición (categoría: {cat})")
    if response.stop_reason == "max_tokens":
        raise RuntimeError("Respuesta truncada (max_tokens); baja BATCH")
    return next(b.text for b in response.content if b.type == "text")


def preparar_ficha(subs, idioma="español latinoamericano neutro", notas="", client=None):
    """Lee todos los subtítulos y devuelve una ficha (markdown) de personajes, trato y glosario."""
    client = client or cliente()
    texto = "\n".join(f"{s['id']}: {s['texto'].replace(chr(10), ' ')}" for s in subs)
    prompt = FICHA_PROMPT.format(idioma=idioma, texto=texto,
                                 notas=f"\nNotas del usuario sobre la obra: {notas}\n" if notas else "")
    return _llamar(client, messages=[{"role": "user", "content": prompt}],
                   output_config={"effort": EFFORT}).strip()


def _pedir(client, system, contexto, bloque):
    partes = []
    if contexto:
        partes.append("Contexto (subtítulos anteriores, ya traducidos; NO los traduzcas de nuevo):\n"
                      + json.dumps(contexto, ensure_ascii=False))
    partes.append("Traduce estos subtítulos:\n"
                  + json.dumps([{"id": s["id"], "texto": s["texto"]} for s in bloque], ensure_ascii=False))
    text = _llamar(
        client,
        # El system (reglas + ficha) es igual en todos los bloques: se cachea.
        system=[{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": "\n\n".join(partes)}],
        output_config={"effort": EFFORT, "format": {"type": "json_schema", "schema": SCHEMA}},
    )
    return {d["id"]: d["texto"] for d in json.loads(text)["subtitulos"]}


def traducir(subs, idioma="español latinoamericano neutro", notas="", client=None, verbose=True, ficha=""):
    """Devuelve una copia de `subs` con el texto traducido. Los tiempos no se tocan."""
    client = client or cliente()
    system = SYSTEM.format(idioma=idioma, chars=LINE_CHARS,
                           notas=f"\nNotas sobre la obra:\n{notas}" if notas else "",
                           ficha=f"\n<ficha_de_la_obra>\n{ficha}\n</ficha_de_la_obra>" if ficha else "")
    out = [dict(s) for s in subs]
    for i in range(0, len(subs), BATCH):
        bloque = subs[i:i + BATCH]
        contexto = [{"id": s["id"], "original": subs[j]["texto"], "traduccion": s["texto"]}
                    for j, s in enumerate(out[max(0, i - CONTEXT):i], start=max(0, i - CONTEXT))]
        pendientes, trad = bloque, {}
        for _ in range(2):                      # 1 reintento solo con los que faltaron
            try:
                trad.update(_pedir(client, system, contexto, pendientes))
            except (anthropic.AuthenticationError, anthropic.PermissionDeniedError, anthropic.BadRequestError):
                raise                           # key o petición inválida: no tiene sentido seguir
            except (RuntimeError, json.JSONDecodeError, StopIteration, anthropic.APIStatusError) as e:
                print(f"  ! bloque {i // BATCH + 1}: {e}")
            pendientes = [s for s in bloque if s["id"] not in trad]
            if not pendientes:
                break
        for k, s in enumerate(bloque):
            if s["id"] in trad:
                out[i + k]["texto"] = _rearmar(trad[s["id"]])
        if pendientes:
            print(f"  ! {len(pendientes)} subtítulos quedaron sin traducir (ids {[s['id'] for s in pendientes][:10]}...)")
        if verbose:
            print(f"  {min(i + BATCH, len(subs))}/{len(subs)}")
    return out


def ficha_de(path, subs, idioma, notas, client):
    """Usa nombre.ficha.md si existe (para poder editarla); si no, la prepara y la guarda."""
    fpath = path[:-4] + ".ficha.md"
    if os.path.exists(fpath):
        print("  usando ficha existente:", fpath)
        return open(fpath, encoding="utf-8").read()
    ficha = preparar_ficha(subs, idioma, notas, client)
    with open(fpath, "w", encoding="utf-8") as f:
        f.write(ficha + "\n")
    print("  ficha guardada en", fpath)
    return ficha


def main():
    import argparse
    global MODEL, EFFORT
    ap = argparse.ArgumentParser(description="Traduce .srt con Claude")
    ap.add_argument("srt", nargs="+")
    ap.add_argument("--idioma", default="español latinoamericano neutro")
    ap.add_argument("--sufijo", default="spa", help="archivo de salida: nombre.<sufijo>.srt")
    ap.add_argument("--notas", default="", help="contexto de la obra: título, época, personajes, glosario")
    ap.add_argument("--sin-ficha", action="store_true", help="no preparar la ficha de personajes")
    ap.add_argument("--model", default=MODEL)
    ap.add_argument("--effort", default=EFFORT)
    a = ap.parse_args()
    MODEL, EFFORT = a.model, a.effort
    client = cliente()
    for path in a.srt:
        out = path[:-4] + f".{a.sufijo}.srt"
        print(path, "→", out)
        subs = leer_srt(path)
        ficha = "" if a.sin_ficha else ficha_de(path, subs, a.idioma, a.notas, client)
        escribir_srt(traducir(subs, a.idioma, a.notas, client, ficha=ficha), out)
    print(f"Tokens ({MODEL}): entrada={USO['input']} salida={USO['output']}")


if __name__ == "__main__":
    main()

## 1. Transcribir

In [ ]:
import importlib, transcribe
importlib.reload(transcribe)

archivos = os.listdir(folder_path)
mp3s = sorted(os.path.splitext(f)[0] for f in archivos if f.lower().endswith('.mp3'))
srts = {f[:-4] for f in archivos if f.lower().endswith('.srt')}
pendientes = [m for m in mp3s if m not in srts]
print(f"{len(pendientes)} mp3 sin subtítulos:", *pendientes, sep="\n- ")

if TRANSCRIBIR and pendientes:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = transcribe.WhisperModel(WHISPER_MODEL, device=device,
                                    compute_type="float16" if device == "cuda" else "int8")
    for nombre in pendientes:
        print("Transcribiendo:", nombre)
        words = transcribe.transcribir(model, os.path.join(folder_path, nombre + ".mp3"), language=IDIOMA_AUDIO)
        transcribe.escribir_srt(transcribe.armar_cues(words), os.path.join(folder_path, nombre + ".srt"))

## 2. Traducir con Claude

In [ ]:
import importlib, translate
from google.colab import userdata
importlib.reload(translate)
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:   # solo si tu key no está asociada a un workspace
    os.environ["ANTHROPIC_WORKSPACE_ID"] = userdata.get("ANTHROPIC_WORKSPACE_ID")
except Exception:
    pass

archivos = os.listdir(folder_path)
originales =[f for f in archivos if f.lower().endswith('.srt') and not f[:-4].endswith(f'.{SUFIJO}')]
pendientes_tr = sorted(f for f in originales if f[:-4] + f'.{SUFIJO}.srt' not in archivos)
print(f"{len(pendientes_tr)} srt sin traducir:", *pendientes_tr, sep="\n- ")

if TRADUCIR:
    for f in pendientes_tr:
        print("Traduciendo:", f)
        path = os.path.join(folder_path, f)
        subs = translate.leer_srt(path)
        # Ficha de personajes/trato/glosario: se guarda como nombre.ficha.md; si la editas, se reutiliza.
        ficha = translate.ficha_de(path, subs, IDIOMA_DESTINO, NOTAS, translate.cliente()) if USAR_FICHA else ""
        subs = translate.traducir(subs, IDIOMA_DESTINO, NOTAS, ficha=ficha)
        translate.escribir_srt(subs, os.path.join(folder_path, f[:-4] + f'.{SUFIJO}.srt'))

## (Opcional) Validar tiempos contra un SRT de referencia
Compara solo tiempos; la referencia puede estar en otro idioma. F1 = cuánto coincide el tiempo en pantalla.

In [ ]:
%%writefile evalsrt.py
"""Compara SOLO tiempos de un SRT generado contra uno de referencia (el idioma puede diferir).

Uso: python evalsrt.py ref.srt hyp.srt [--max SEG]
"""
import re, sys, argparse
import numpy as np

TS = re.compile(r"(\d+):(\d+):(\d+)[,.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,.](\d+)")


def load(path, tmax=None):
    cues = []
    for m in TS.finditer(open(path, "rb").read().decode("utf-8", "replace")):
        g = list(map(int, m.groups()))
        s = g[0] * 3600 + g[1] * 60 + g[2] + g[3] / 1000
        e = g[4] * 3600 + g[5] * 60 + g[6] + g[7] / 1000
        if tmax is None or s < tmax:
            cues.append((s, min(e, tmax) if tmax else e))
    return cues


def mask(cues, n, res):
    m = np.zeros(n, bool)
    for s, e in cues:
        m[int(s / res):int(e / res)] = True
    return m


def evaluate(ref, hyp, tmax, res=0.01):
    n = int(tmax / res) + 1
    R, H = mask(ref, n, res), mask(hyp, n, res)
    tp = (R & H).sum()
    prec, rec = tp / max(H.sum(), 1), tp / max(R.sum(), 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    hs = np.array([s for s, _ in hyp]); he = np.array([e for _, e in hyp])
    ds = np.array([np.min(np.abs(hs - s)) for s, _ in ref]) if len(hyp) else np.array([np.inf])
    de = np.array([np.min(np.abs(he - e)) for _, e in ref]) if len(hyp) else np.array([np.inf])
    long = sum(1 for s, e in hyp if e - s > 7)
    return dict(n_ref=len(ref), n_hyp=len(hyp), precision=prec, recall=rec, f1=f1,
                start_med=np.median(ds), start_05=(ds <= 0.5).mean(), start_1=(ds <= 1).mean(),
                end_med=np.median(de), end_1=(de <= 1).mean(), largos_7s=long)


def fmt(name, r):
    return (f"{name:28s} cues={r['n_hyp']:4d}/{r['n_ref']:<4d} F1={r['f1']:.3f} "
            f"(P={r['precision']:.2f} R={r['recall']:.2f}) | inicio: mediana={r['start_med']:.2f}s "
            f"≤0.5s={r['start_05']:.0%} ≤1s={r['start_1']:.0%} | fin: mediana={r['end_med']:.2f}s "
            f"≤1s={r['end_1']:.0%} | >7s={r['largos_7s']}")


def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("ref"); ap.add_argument("hyp", nargs="+")
    ap.add_argument("--max", type=float, default=None, help="evaluar solo hasta este segundo")
    a = ap.parse_args()
    tmax = a.max or max(e for _, e in load(a.ref))
    ref = load(a.ref, tmax)
    for h in a.hyp:
        print(fmt(h.split("/")[-1], evaluate(ref, load(h, tmax), tmax)))


if __name__ == "__main__":
    main()

In [ ]:
import evalsrt
ref_srt = '/content/drive/MyDrive/<ruta>/example.srt'
hyp_srt = '/content/drive/MyDrive/<ruta>/example_gen.srt'
TMAX = None   # p.ej. 1200 para evaluar solo los primeros 20 min
tmax = TMAX or max(e for _, e in evalsrt.load(ref_srt))
print(evalsrt.fmt('generado', evalsrt.evaluate(evalsrt.load(ref_srt, tmax), evalsrt.load(hyp_srt, tmax), tmax)))